# Lab 2: Vectorless RAG — Advanced Scenarios

## Setup

### Install Dependencies

In [ ]:
# Install PageIndex for vectorless hierarchical retrieval, LangChain Groq for the LLM, and requests for downloading the file.
!pip install pageindex langchain-groq requests dotenv 

### Import Libraries

In [ ]:
# Import core modules
import os
import time
import requests
import re

# Import PageIndex for vectorless retrieval
from pageindex import PageIndexClient
import pageindex.utils as utils
from dotenv import load_dotenv

# Import LangChain's Groq wrapper
from langchain_groq import ChatGroq

### Setup API Keys

In [28]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

# Initialize the PageIndex Client
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

---
## Load and Parse the PDF

### Define PDF Path

In [13]:
import os

# Point to the local PDF document you already have
PDF_PATH = "data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf"

# Let's add a quick beginner-friendly check to make sure the file exists where we expect it
if os.path.exists(PDF_PATH):
    print(f"Success: Found the document at '{PDF_PATH}'")
else:
    print(f"Error: Could not find the document. Please make sure the 'data' folder is in the same directory as this notebook.")

Success: Found the document at 'data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf'


### Submit and Index Document (Tree Construction)

In [ ]:
# Submit your local document to PageIndex. 
# It reads your financial document and organizes it into a logical tree (Sections, Tables, Text).
doc_info = pi_client.submit_document(PDF_PATH)
doc_id = doc_info["doc_id"]

print(f"Document Submitted. Tracking ID: {doc_id}")

# Polling loop: Wait for the document tree to finish building
print("Waiting for the document to be indexed...")
while not pi_client.is_retrieval_ready(doc_id):
    print("Still processing... checking again in 5 seconds.")
    time.sleep(5)

print("Indexing Complete! The hierarchical tree is ready for multi-hop retrieval.")

Document Submitted. Tracking ID: pi-cmrmwiexy00l101o46ca8q8c0
Waiting for the document to be indexed...
Still processing... checking again in 5 seconds.
Still processing... checking again in 5 seconds.
Still processing... checking again in 5 seconds.
Indexing Complete! The hierarchical tree is ready for multi-hop retrieval.


In [132]:
tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
print("Document Tree Structure:")
utils.print_tree(tree)

Document Tree Structure:
[{'title': 'Century Communities Reports First Quarte...',
  'node_id': '0000',
  'summary': 'Century Communities reported its Q1 2025...'},
 {'title': 'First Quarter 2025 Results',
  'node_id': '0001',
  'prefix_summary': 'The report details the Q1 2025 financial...',
  'nodes': [{'title': 'Balance Sheet and Liquidity',
             'node_id': '0002',
             'summary': 'In Q1 2025, the company maintained a str...'},
            {'title': 'Full Year 2025 Outlook',
             'node_id': '0003',
             'summary': '## Full Year 2025 Outlook\n\nScott Dixon, ...'},
            {'title': 'Webcast and Conference Call',
             'node_id': '0004',
             'summary': 'Century Communities will host a webcast ...'},
            {'title': 'About Century Communities',
             'node_id': '0005',
             'summary': 'Century Communities, Inc. is a prominent...'},
            {'title': 'Non-GAAP Financial Measures',
             'node_id': '0006'

### Initialize OpenRouter LLM

In [142]:
# Initialize the Large Language Model via OpenRouter
# OpenRouter gives you access to multiple models. Here we use an open model as an example.
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0.0,
    max_tokens=300  
)

### Define Retrieval Function (With Explainability Tracking)

In [143]:
def retrieve_from_pageindex(query, doc_id, top_k=5):
    """
    Searches the document tree for the given query.
    Every piece of context returned here is tagged with:
      - which hop it was (1st match, 2nd match, ...)
      - the section title
      - the node id (the tree's unique ID for that section)
      - the page number(s) the text came from
    This metadata is what lets us later explain WHY a node was chosen.
    """
    response = pi_client.submit_query(doc_id=doc_id, query=query)
    retrieval_id = response.get("retrieval_id")

    if not retrieval_id:
        return []

    while True:
        retrieval = pi_client.get_retrieval(retrieval_id)
        status = retrieval.get("status")
        if status == "completed":
            break
        elif status == "failed":
            return []
        time.sleep(1)

    nodes = retrieval.get("retrieved_nodes", [])
    hops = []

    for index, node in enumerate(nodes[:top_k]):
        node_name = node.get("title") or f"Section {index + 1}"
        node_id = node.get("id", "unknown")   
        relevant_contents = node.get("relevant_contents", [])

        section_text = []
        page_numbers = []
        for group in relevant_contents:
            for item in group:
                content = item.get("relevant_content")
                if content:
                    section_text.append(content)

                # page number is embedded in a string like "<physical_index_6>"
                raw_page = item.get("physical_index", "")
                match = re.search(r"(\d+)", raw_page) if isinstance(raw_page, str) else None
                if match:
                    page_num = int(match.group(1))
                    if page_num not in page_numbers:
                        page_numbers.append(page_num)

        hops.append({
            "hop_number": index + 1,
            "section": node_name,
            "node_id": node_id,
            "pages": page_numbers,
            "text": "\n".join(section_text)
        })

    return hops

In [144]:
def vectorless_rag(query, doc_id):
    hops = retrieve_from_pageindex(query, doc_id)

    if not hops:
        return "No relevant context found.", [], ""

    labeled_context = "\n\n".join(h["text"] for h in hops)

    prompt = f"""
You are a financial analyst. Answer the question below using only the context provided.
Give a direct, interpreted answer in plain language, the way an analyst would explain it
to someone in a meeting — not a mechanical calculation. Consider seasonality and other
relevant signals in the context (like backlog or order trends) rather than just scaling
one quarter's number by 4.

Context:
{labeled_context}

Question: {query}
"""

    response = llm.invoke(prompt)
    final_answer = response.content

    return final_answer, hops, labeled_context

### Run Query and Show the Tracking

In [151]:
# The question we want to ask
query = "How much did GAAP net income grow or shrink from Q1 2024 to Q1 2025? Separately, what was adjusted net income (not adjusted EBITDA) for both periods, and does it tell a different story?"

In [155]:
print(f"Question: {query}\n")
print("Navigating document tree and generating answer...\n")

# Run the pipeline
final_answer, hops, labeled_context = vectorless_rag(query, doc_id)

# --- SECTIONS SEARCHED (RETRIEVAL TRACE) ---
print("--- SECTIONS SEARCHED (RETRIEVAL TRACE) ---")
for hop in hops:
    print(f"Hop {hop['hop_number']}: {hop['section']}")

# --- FINAL ANSWER ---
print("\n--- FINAL ANSWER ---")
print(final_answer)

Question: How much did GAAP net income grow or shrink from Q1 2024 to Q1 2025? Separately, what was adjusted net income (not adjusted EBITDA) for both periods, and does it tell a different story?

Navigating document tree and generating answer...

--- SECTIONS SEARCHED (RETRIEVAL TRACE) ---
Hop 1: First Quarter 2025 Results
Hop 2: Adjusted Net Income and Adjusted Diluted Earnings Per Share
Hop 3: EBITDA and Adjusted EBITDA

--- FINAL ANSWER ---
Let's break down the key points here. 

First, looking at GAAP net income, we see a significant decline from Q1 2024 to Q1 2025. The actual decrease is $24,948 thousand, which translates to a 38.8% drop from $64,332 thousand in Q1 2024 to $39,384 thousand in Q1 2025. This is a substantial decline, indicating a challenging quarter for the company.

Now, let's consider adjusted net income. Here, we see a decline as well, but the absolute decrease is slightly larger at $29,190 thousand. This translates to a decrease from $71,430 thousand in Q1 2024

In [156]:
print("\n--- EXPLAINABILITY ---")
for hop in hops:
    pages = ", ".join(str(p) for p in hop["pages"]) if hop["pages"] else "unknown"
    was_used = hop["hop_number"] 
    status = "USED in answer" if was_used else "retrieved but NOT used"

    print(f"\nHop {hop['hop_number']}: \"{hop['section']}\"")
    print(f"node_id: {hop['node_id']} | page(s): {pages} | {status}")

    # Ask the LLM directly, right here, why this hop is relevant --
    # no helper function, built inline for each hop.
    # Note: this is a generated explanation, not PageIndex's internal
    # scoring (the API doesn't expose that), but it's grounded in the
    # actual retrieved content, not made up.
    explain_prompt = f"""
In 3-4 short lines, explain why the section below is relevant to the question.
Be specific -- mention the actual numbers or facts in the section that connect to the question.
Do not repeat the question. Do not add extra commentary.

Question: {query}

Section title: {hop['section']}
Section content: {hop['text']}
"""
    explanation_response = llm.invoke(explain_prompt)
    print(f"Why: {explanation_response.content.strip()}")


--- EXPLAINABILITY ---

Hop 1: "First Quarter 2025 Results"
node_id: 0001 | page(s): 1 | USED in answer
Why: This section is relevant to the question because it provides GAAP net income for Q1 2025 as $39.4 million. It also provides adjusted net income for Q1 2025 as $42.2 million.

Hop 2: "Adjusted Net Income and Adjusted Diluted Earnings Per Share"
node_id: 0010 | page(s): 8 | USED in answer
Why: The section is relevant to the question because it provides specific numbers for GAAP net income and adjusted net income in Q1 2024 and Q1 2025, showing a decrease of $24,948 thousand for GAAP net income and $29,190 thousand for adjusted net income.

Hop 3: "EBITDA and Adjusted EBITDA"
node_id: 0012 | page(s): 10 | USED in answer
Why: The section is relevant to the question because it provides specific numbers for GAAP net income and adjusted EBITDA for Q1 2024 and Q1 2025. The GAAP net income decreased by 38.8% from $64,332 to $39,384.
